# Fine-tuning de ResNet-50 pour XAgriScan
Ce notebook configure et entraîne un modèle **ResNet-50** pré-entraîné sur les données d'images de feuilles de pommiers en utilisant **PyTorch**.

In [2]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader


In [3]:
# Paramètres
data_dir = '../data'
batch_size = 37
num_epochs = 15
num_classes = 6 

# Transformations des données (Data Augmentation pour l'entraînement)
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Chargement des datasets
image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True if x == 'train' else False, num_workers=4)
    for x in ['train', 'val']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Classes : {class_names}")
print(f"Tailles des datasets : {dataset_sizes}")

Classes : ['apple_frogeye_leaf_spot', 'apple_leaf_healthy', 'apple_mosaic_leaf', 'apple_powdery_mildew_leaf', 'apple_rust_leaf', 'apple_scab_leaf']
Tailles des datasets : {'train': 1038, 'val': 231}


In [5]:
# Chargement du modèle ResNet-50 pré-entraîné
model = models.resnet50(pretrained=True)

# 1. CONGÉLATION (Freezing) des paramètres de base
# On gèle toutes les couches (les poids de la partie convolutionnelle ne seront pas mis à jour)
for param in model.parameters():
    param.requires_grad = False

# 2. DROP-OUT : Remplacement et ajout de Dropout pour la dernière couche
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5), # 50% de dropout pour réduire le sur-apprentissage
    nn.Linear(num_ftrs, num_classes)
)

model = model.to(device)

# Définition de la fonction de perte
criterion = nn.CrossEntropyLoss()

# On n'optimise QUE les paramètres de la dernière couche (puisqu'on a gelé le reste)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Optionnel : Réduire le learning rate toutes les 7 epochs
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
# Fonction d'entraînement avec EARLY STOPPING
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    epochs_no_improve = 0 # Compteur pour le Early Stopping

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Chaque epoch a une phase d'entraînement et de validation
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Mode entraînement
            else:
                model.eval()   # Mode évaluation

            running_loss = 0.0
            running_corrects = 0

            # Itération sur les données
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimisation si phase d'entraînement
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistiques
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Copie du meilleur modèle et gestion de l'Early Stopping
            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0 # On reset le compteur car on a une amélioration
                else:
                    epochs_no_improve += 1
                    
        print(f'Epochs sans amélioration de validation: {epochs_no_improve}')
        
        # 3. EARLY STOPPING
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping détecté à l'epoch {epoch}! La précision ne s'améliore plus depuis {patience} epochs.")
            break # On arrête l'entraînement

        print()

    time_elapsed = time.time() - since
    print(f'Entraînement terminé en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Meilleure précision en validation: {best_acc:4f}')

    # Charger les meilleurs poids
    model.load_state_dict(best_model_wts)
    return model

# Lancement de l'entraînement avec patience de 4
model_ft = train_model(model, criterion, optimizer, exp_lr_scheduler, num_epochs=num_epochs, patience=4)

In [ ]:
# Sauvegarde du modèle entraîné
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)
save_path = os.path.join(models_dir, 'resnet50_xagriscan.pth')

torch.save(model_ft.state_dict(), save_path)
print(f"Modèle sauvegardé avec succès dans : {save_path}")